# VLA Hallucination Mitigation
## Mitigating Hallucination and Perception Errors in Vision-Language Agents

**Author**: Vedhagiri Alagesan — Heriot-Watt University  
**Supervisor**: Dr. Oliver Lemon  

### Pipeline Overview
1. **Detector Grounding** — Simulated YOLOv8 object detection anchors the VLM to observed objects
2. **Multi-View Consistency** — 4 camera viewpoints (front, left, right, overhead) to filter false detections
3. **Chain-of-Thought (CoT)** — Structured reasoning: `[Observation] → [Multi-View Check] → [Reasoning] → [Verification] → [Answer]`
4. **GRPO Training** — Group Relative Policy Optimization with hallucination-aware rewards
5. **Self-Verification** — Post-generation checks against detector and multi-view evidence

### Training Phases
- **Phase 1 (SFT)**: Supervised fine-tuning on CoT-formatted data to teach response structure
- **Phase 2 (GRPO)**: Reinforcement learning with hallucination penalties to reduce errors

### Evaluation
- Ablation study (Table 3 from dissertation)
- Comparison with POPE, ICD, MIHBench baselines

---
## 1. Setup

In [ ]:
# Clone repository and install dependencies
import os

REPO_URL = "https://github.com/Gsam0612/vla-hallucination-mitigation.git"
PROJECT_DIR = "/content/vla-hallucination-mitigation"

if not os.path.exists(PROJECT_DIR):
    # For Colab: clone from GitHub
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Core imports
import sys
import json
import random
import numpy as np
import torch
from pathlib import Path

# Add project root to path (handles both Colab and local)
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TrainingConfig, RewardConfig, GRPOConfig, MultiViewConfig, EvalConfig
from src.objects import AI2THOR_OBJECTS, HAZARD_OBJECTS
from src.reward import HallucinationReward
from src.multi_view import MultiViewConsistency
from src.scene_generator import generate_scene, generate_scenes
from src.data_generator import generate_dataset, generate_training_sample
from src.dataset import VLADataset, collate_fn
from src.grpo_trainer import GRPOTrainer
from src.evaluation import VLAEvaluator, format_ablation_table, format_comparison_table
from src.inference import VLAInferencePipeline

# Seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 2. Configuration

All hyperparameters follow Section 4 of the dissertation.

In [ ]:
# Training configuration
# NOTE: 2000 samples is sufficient for SFT on synthetic data.
# On A100, increase batch_size to 4 for ~4x speedup.
config = TrainingConfig(
    model_name="llava-hf/llava-1.5-7b-hf",
    output_dir="./outputs",
    num_training_samples=2000,    # 2000 is enough for synthetic CoT data
    num_eval_samples=200,
    seed=SEED,
    # LoRA
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # SFT Phase
    max_length=512,               # Reduced from 1024 — CoT answers rarely exceed 400 tokens
    sft_epochs=2,
    sft_batch_size=4,             # 4 on A100 (use 2 on T4)
    sft_learning_rate=2e-4,
    sft_warmup_steps=50,
)

# Reward penalties/bonuses (Dissertation Section 4.5)
config.reward = RewardConfig(
    object_existence_penalty=-1.0,
    misidentification_penalty=-0.8,
    attribute_error_penalty=-0.5,
    spatial_relation_penalty=-0.6,
    correct_object_bonus=0.2,
    correct_count_bonus=0.3,
    grounded_response_bonus=0.5,
    safety_awareness_bonus=0.4,
    multi_view_consistency_bonus=0.3,
    cot_quality_bonus=0.2,
)

# Multi-view (Section 4.4.2)
config.multi_view = MultiViewConfig(
    num_views=4,
    view_angles={'front': 0, 'left': -45, 'right': 45, 'overhead': -90},
    consistency_threshold=0.5,
    aggregation='majority',
)

# GRPO (Section 4.5)
config.grpo = GRPOConfig(
    num_candidates=4,
    temperature=0.8,
    max_new_tokens=300,
    kl_coeff=0.1,
    clip_range=0.2,
    grpo_epochs=3,
    grpo_mini_batch_size=2,
    learning_rate=2e-5,
    gradient_accumulation_steps=4,
)

os.makedirs(config.output_dir, exist_ok=True)
print("Configuration set!")
print(f"  Model: {config.model_name}")
print(f"  LoRA r={config.lora_r}, alpha={config.lora_alpha}")
print(f"  SFT epochs: {config.sft_epochs}")
print(f"  GRPO epochs: {config.grpo.grpo_epochs}, K={config.grpo.num_candidates}")
print(f"  Training samples: {config.num_training_samples}")

---
## 3. Generate Training Data

Each sample includes:
- AI2-THOR scene (objects with positions, colors, relations)
- Simulated YOLOv8 detector output (with confidence scores)
- Multi-view consistency check (4 views)
- **CoT-formatted answer**: `[Observation] → [Multi-View Check] → [Reasoning] → [Verification] → [Answer]`
- Ground truth for reward computation

In [ ]:
# Initialize multi-view checker
mv_checker = MultiViewConsistency(config.multi_view)

# Generate training dataset
print("Generating training dataset with CoT format...")
training_data = generate_dataset(
    n=config.num_training_samples,
    mv_checker=mv_checker,
)
print(f"Generated {len(training_data)} training samples")

# Generate evaluation dataset
print("\nGenerating evaluation dataset...")
eval_data = generate_dataset(
    n=config.num_eval_samples,
    mv_checker=mv_checker,
)
print(f"Generated {len(eval_data)} evaluation samples")

# Show example
example = training_data[0]
print("\n" + "="*60)
print("EXAMPLE TRAINING SAMPLE")
print("="*60)
print(f"Question type: {example.get('question_type', 'N/A')}")
print(f"Room: {example['scene']['room_type']}")
print(f"Objects: {', '.join(example['scene']['object_list'])}")
print(f"\nDetector prompt:\n{example['detector_prompt'][:200]}...")
print(f"\nCoT answer (first 400 chars):\n{example['answer'][:400]}...")

In [ ]:
# Inspect data distribution
from collections import Counter

q_types = Counter(d.get('question_type', 'unknown') for d in training_data)
rooms = Counter(d['scene']['room_type'] for d in training_data)
complexities = Counter(d['scene']['complexity'] for d in training_data)

print("Question types:")
for qt, count in q_types.most_common():
    print(f"  {qt}: {count}")

print(f"\nRooms: {dict(rooms)}")
print(f"Complexities: {dict(complexities)}")

# Verify CoT format
cot_sections = ['[Observation]', '[Multi-View Check]', '[Reasoning]', '[Verification]', '[Answer]']
cot_complete = sum(1 for d in training_data if all(s in d['answer'] for s in cot_sections))
print(f"\nSamples with complete CoT format: {cot_complete}/{len(training_data)} ({cot_complete/len(training_data)*100:.1f}%)")

---
## 4. Load Model

LLaVA-1.5-7B with:
- 4-bit NF4 quantization (fits in ~5GB VRAM)
- LoRA adapters (r=16, alpha=32) on attention + MLP projections

In [ ]:
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 4-bit quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {config.model_name}...")
model = LlavaForConditionalGeneration.from_pretrained(
    config.model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(config.model_name)

print(f"Model loaded!")
print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Apply LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=config.target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"\nGPU Memory after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## 5. Phase 1: Supervised Fine-Tuning (SFT)

Train the model on CoT-formatted responses to learn the structured reasoning format.

**Key features:**
- Proper **label masking**: only compute loss on ASSISTANT tokens (not system prompt or user turn)
- Gradient checkpointing for memory efficiency
- Paged AdamW 8-bit optimizer

In [ ]:
from transformers import TrainingArguments, Trainer

# Create dataset with label masking
train_dataset = VLADataset(training_data, processor, max_length=config.max_length)
print(f"Training dataset: {len(train_dataset)} samples")

# Verify label masking
sample = train_dataset[0]
masked_count = (sample['labels'] == -100).sum().item()
total_tokens = (sample['labels'] != -100).sum().item()
print(f"Label masking check: {masked_count} masked tokens, {total_tokens} training tokens")
print(f"Masking ratio: {masked_count/(masked_count+total_tokens)*100:.1f}% masked (prompt + padding)")

In [ ]:
# SFT Training Arguments
# A100: batch=4, grad_accum=4 → effective batch=16, ~125 steps/epoch → ~5min/epoch
# T4:   batch=2, grad_accum=8 → effective batch=16, ~125 steps/epoch → ~45min/epoch
sft_args = TrainingArguments(
    output_dir=os.path.join(config.output_dir, "sft"),
    num_train_epochs=config.sft_epochs,
    per_device_train_batch_size=config.sft_batch_size,
    gradient_accumulation_steps=4,      # 4 on A100 (use 8 on T4)
    learning_rate=config.sft_learning_rate,
    weight_decay=0.01,
    warmup_steps=config.sft_warmup_steps,
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

effective_batch = config.sft_batch_size * 4
total_steps = (len(train_dataset) // effective_batch) * config.sft_epochs
print("Starting Phase 1: SFT Training...")
print(f"  Epochs: {config.sft_epochs}")
print(f"  Batch size: {config.sft_batch_size} (effective: {effective_batch})")
print(f"  Total steps: ~{total_steps}")
print(f"  Learning rate: {config.sft_learning_rate}")

sft_result = trainer.train()

print(f"\nSFT Training Complete!")
print(f"  Train loss: {sft_result.training_loss:.4f}")
print(f"  GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## 6. Phase 2: GRPO Training

**Group Relative Policy Optimization** (Dissertation Section 4.5):
1. For each prompt, generate **K=4 candidate** responses
2. Score each with the **hallucination-aware reward** function
3. Compute **group-relative advantages**: $A_i = \frac{R_i - \mu_R}{\sigma_R}$
4. Update policy: $\mathcal{L} = -\mathbb{E}[A_i \cdot \log \pi_\theta(y_i | x)]$

**Reward components:**
- Object existence penalty (-1.0)
- Misidentification penalty (-0.8)
- Attribute error penalty (-0.5)
- Spatial relation penalty (-0.6)
- Correct object bonus (+0.2)
- Multi-view consistency bonus (+0.3)
- CoT quality bonus (+0.2)
- Safety awareness bonus (+0.4)

In [ ]:
# Initialize reward function and GRPO trainer
reward_fn = HallucinationReward(config.reward)

grpo_trainer = GRPOTrainer(
    model=model,
    processor=processor,
    reward_fn=reward_fn,
    config=config.grpo,
    training_config=config,
)

# Use subset for GRPO (generation is expensive)
grpo_data = random.sample(training_data, min(500, len(training_data)))
print(f"GRPO training on {len(grpo_data)} samples")
print(f"  K={config.grpo.num_candidates} candidates per prompt")
print(f"  Total generations per epoch: {len(grpo_data) * config.grpo.num_candidates}")

In [ ]:
# Run GRPO training
print("Starting Phase 2: GRPO Training...")
print("This will take a while due to generation + scoring + policy updates.\n")

grpo_metrics = grpo_trainer.train_grpo(
    training_data=grpo_data,
    num_epochs=config.grpo.grpo_epochs,
    save_dir=config.output_dir,
)

# Print final metrics
print("\n" + "="*60)
print("GRPO Training Summary")
print("="*60)
if grpo_metrics:
    for key, value in grpo_metrics[-1].items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")

# Save GRPO training log
grpo_trainer.save_training_log(
    os.path.join(config.output_dir, "grpo_training_log.json")
)

In [ ]:
# Visualize GRPO training progress
import matplotlib.pyplot as plt

if grpo_metrics:
    steps = range(len(grpo_metrics))
    rewards = [m.get('mean_reward', 0) for m in grpo_metrics]
    h_rates = [m.get('mean_hallucination_rate', 0) for m in grpo_metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, rewards, 'b-', linewidth=2)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Mean Reward')
    ax1.set_title('GRPO Training: Reward')
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps, h_rates, 'r-', linewidth=2)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Hallucination Rate')
    ax2.set_title('GRPO Training: Hallucination Rate')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'grpo_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Training curves saved.")

---
## 7. Evaluation

### 7.1 Ablation Study (Dissertation Table 3)

| Configuration | Components |
|---|---|
| baseline | Raw LLaVA (no enhancements) |
| detection_only | + YOLOv8 detector grounding |
| detection_multiview | + Multi-view consistency (4 views) |
| detection_cot | + Chain-of-thought reasoning |
| full_no_grpo | + All components, SFT only |
| full_with_grpo | + All components + GRPO training |

In [ ]:
# Setup evaluator
evaluator = VLAEvaluator(
    model=model,
    processor=processor,
    reward_fn=reward_fn,
    mv_checker=mv_checker,
)

# Run ablation study
print("Running ablation study...")
print("This evaluates each component's contribution to hallucination reduction.\n")

ablation_results = evaluator.run_ablation(num_scenes_per_config=100)

# Display results
print("\n" + format_ablation_table(ablation_results))

In [ ]:
# Visualize ablation results
import matplotlib.pyplot as plt
import numpy as np

configs = list(ablation_results.keys())
h_rates = [ablation_results[c].get('mean_hallucination_rate', 0) for c in configs]
recalls = [ablation_results[c].get('mean_recall', 0) for c in configs]
precisions = [ablation_results[c].get('mean_precision', 0) for c in configs]

x = np.arange(len(configs))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width, h_rates, width, label='Hallucination Rate', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x, recalls, width, label='Recall', color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, precisions, width, label='Precision', color='#3498db', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Ablation Study Results (Dissertation Table 3)')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in configs], fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Comparison with Existing Methods

Compare our full system against published baselines:
- **POPE** (Li et al., 2023): Polling-based Object Probing
- **ICD** (Leng et al., 2024): Induced Contrastive Decoding
- **MIHBench** (Chen et al., 2025): Multi-modal Image Hallucination Benchmark

In [ ]:
# Compare with published baselines
our_results = ablation_results.get('full_with_grpo', ablation_results.get('full_no_grpo', {}))
comparison = VLAEvaluator.compare_with_baselines(our_results)

print(format_comparison_table(comparison))

---
## 8. Inference Demo

Test the trained model on individual scenes.

In [ ]:
# Create inference pipeline
pipeline = VLAInferencePipeline(
    model=model,
    processor=processor,
    mv_checker=mv_checker,
    reward_fn=reward_fn,
)

# Test on a kitchen scene
test_scene = generate_scene('simple', 'Kitchen')
print(f"Scene: {test_scene.room_type} ({test_scene.complexity})")
print(f"Objects: {', '.join(test_scene.get_object_list())}")
print(f"Hazards: {', '.join(test_scene.hazards) or 'None'}")

# Run inference with full pipeline
result = pipeline.infer(
    question="What do you see in front of you? Are there any safety concerns?",
    scene_objects=test_scene.objects,
    use_multiview=True,
    use_cot=True,
)

print("\n" + "="*60)
print("INFERENCE RESULT")
print("="*60)
print(f"\nFull CoT Response:\n{result['full_response']}")
print(f"\nFinal Answer: {result['final_answer']}")
print(f"\nVerification:")
print(f"  Reliable: {result['verification']['is_reliable']}")
print(f"  Confidence: {result['verification']['confidence']:.2%}")
print(f"  Grounded objects: {result['verification']['grounded']}")
print(f"  Ungrounded objects: {result['verification']['ungrounded']}")

In [ ]:
# Test on multiple scenes
test_configs = [
    ('simple', 'Kitchen', 'What objects do you see?'),
    ('cluttered', 'LivingRoom', 'Describe the room in detail.'),
    ('hazard', 'Kitchen', 'Are there any safety hazards?'),
    ('simple', 'Bedroom', 'What is on the table?'),
]

for complexity, room, question in test_configs:
    scene = generate_scene(complexity, room)
    result = pipeline.infer(
        question=question,
        scene_objects=scene.objects,
        use_multiview=True,
        use_cot=True,
    )
    print(f"\n{'='*60}")
    print(f"Scene: {room} ({complexity}) | Q: {question}")
    print(f"Objects: {', '.join(scene.get_object_list())}")
    print(f"Answer: {result['final_answer'][:200]}")
    print(f"Reliable: {result['verification']['is_reliable']} | Confidence: {result['verification']['confidence']:.2%}")

---
## 9. Save Model & Export

Save the trained LoRA adapters, processor, and training configuration.

In [ ]:
# Save model
save_dir = os.path.join(config.output_dir, "final_model")
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

# Save training config
training_info = {
    'base_model': config.model_name,
    'lora_r': config.lora_r,
    'lora_alpha': config.lora_alpha,
    'training_samples': len(training_data),
    'sft_epochs': config.sft_epochs,
    'grpo_epochs': config.grpo.grpo_epochs,
    'grpo_candidates_k': config.grpo.num_candidates,
    'hallucination_types': [
        'object_existence', 'misidentification',
        'attribute', 'spatial_relation',
    ],
    'reward_config': {
        'object_existence_penalty': config.reward.object_existence_penalty,
        'misidentification_penalty': config.reward.misidentification_penalty,
        'attribute_error_penalty': config.reward.attribute_error_penalty,
        'spatial_relation_penalty': config.reward.spatial_relation_penalty,
    },
    'pipeline_components': [
        'yolov8_detector_grounding',
        'multi_view_consistency_4_views',
        'chain_of_thought_reasoning',
        'self_verification',
        'grpo_rl_training',
    ],
}

with open(os.path.join(save_dir, 'training_config.json'), 'w') as f:
    json.dump(training_info, f, indent=2)

print(f"Model saved to: {save_dir}")
print(f"Files: {os.listdir(save_dir)}")

In [ ]:
# Optional: Push to Hugging Face Hub
PUSH_TO_HUB = False  # Set to True and update repo_id to push

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # Enter your HF token

    repo_id = "YOUR_USERNAME/vla-hallucination-mitigation"  # <-- Update this
    model.push_to_hub(repo_id)
    processor.push_to_hub(repo_id)
    print(f"Pushed to: https://huggingface.co/{repo_id}")

---
## Summary

### Key Results
- **Detector grounding** anchors VLM outputs to perceived objects, reducing hallucinated entities
- **Multi-view consistency** (4 viewpoints) filters false detections through cross-view verification
- **Chain-of-Thought** structured reasoning provides transparent, verifiable inference
- **GRPO training** teaches the model to learn from its own mistakes via hallucination-aware rewards
- **Self-verification** catches remaining errors by checking generated text against evidence

### Ablation Insights
Each component contributes incrementally to hallucination reduction, with GRPO providing the final boost by learning from error patterns during training.

### Limitations
- Trained on simulated AI2-THOR scenes (sim-to-real gap for deployment)
- CoT reasoning patterns transfer well, but visual grounding may need domain adaptation
- Detector simulation approximates YOLOv8; real deployment would use actual detection model